# Cargue de Shapefiles de Inundaciones en Colombia\n\nShapefiles del fenómeno **La Niña** a escala 1:100.000 para los años 1988, 2000, 2011 y 2012.\n\n**Fuente:** IGAC / IDEAM  \n**CRS:** EPSG:4686 (MAGNA-SIRGAS)

In [ ]:
import geopandas as gpd

base = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores"

inundacion_1988 = gpd.read_file(rf"{base}\shape 1988\Inundacion_Niña_100k_1988.shp")
inundacion_2000 = gpd.read_file(rf"{base}\shape 2000\Inundacion_Niña_100k_2000.shp")
inundacion_2011 = gpd.read_file(rf"{base}\shape 2011\Inundacion_Niña_100k_2011.shp")
inundacion_2012 = gpd.read_file(rf"{base}\shape 2012\Inundacion_Niña_100k_2012.shp")

for año, gdf in [(1988, inundacion_1988), (2000, inundacion_2000), (2011, inundacion_2011), (2012, inundacion_2012)]:
    print(f"--- {año} ---")
    print(f"  Filas: {len(gdf)} | CRS: {gdf.crs} | Geometría: {gdf.geom_type.unique().tolist()}")


In [ ]:
# Cargar shapefile de municipios (polígonos)
# El .prj está como .prj.txt, por lo que geopandas no lo lee automáticamente → se asigna manualmente
base_mun = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\shape_mun_col"
municipios = gpd.read_file(rf"{base_mun}\Muni.shp")
municipios = municipios.set_crs(epsg=4686)  # MAGNA-SIRGAS, mismo que los shapefiles de inundación

print(f"Municipios cargados: {len(municipios)} | CRS: {municipios.crs}")
municipios[['MunCodigo', 'MunNombre']].head()


In [ ]:
import pandas as pd

# CRS de referencia: EPSG:4686 (MAGNA-SIRGAS), ya asignado a municipios y a los floods
crs_ref = municipios.crs  # ahora es EPSG:4686 y no es None

floods = {
    1988: inundacion_1988.to_crs(crs_ref),
    2000: inundacion_2000.to_crs(crs_ref),
    2011: inundacion_2011.to_crs(crs_ref),
    2012: inundacion_2012.to_crs(crs_ref),
}

# Spatial join: qué municipios intersectan áreas inundadas en cada año
muns_afectados = {}
for año, flood in floods.items():
    joined = gpd.sjoin(
        municipios[['MunCodigo', 'MunNombre', 'geometry']],
        flood[['geometry']],
        how='inner',
        predicate='intersects'
    )
    muns_afectados[año] = set(joined['MunCodigo'].unique())
    print(f"{año}: {len(muns_afectados[año])} municipios afectados")

# Construir tabla binaria (todos los municipios)
tabla = municipios[['MunCodigo', 'MunNombre']].drop_duplicates().copy()
for año in [1988, 2000, 2011, 2012]:
    tabla[str(año)] = tabla['MunCodigo'].isin(muns_afectados[año]).astype(int)

# Frecuencia: proporción de años en que el municipio fue afectado
tabla['frecuencia'] = tabla[['1988', '2000', '2011', '2012']].mean(axis=1).round(2)

tabla = tabla.sort_values('frecuencia', ascending=False).reset_index(drop=True)
tabla = tabla.rename(columns={'MunCodigo': 'cod_divipola', 'MunNombre': 'municipio'})

print(f"\nMunicipios afectados al menos una vez: {(tabla['frecuencia'] > 0).sum()} / {len(tabla)}")
tabla[tabla['frecuencia'] > 0].head(20)
